# 0. Problem
## 197. Rising Temperature — Easy
Return weather row IDs whose temperature is higher than the temperature exactly one day earlier.

Official: https://leetcode.com/problems/rising-temperature/

# 1. Setup

In [ ]:
import pandas as pd
weather_rows=[(1,"2015-01-01",10),(2,"2015-01-02",25),(3,"2015-01-03",20),(4,"2015-01-04",30),(5,"2015-01-06",40)]
weather_pd=pd.DataFrame(weather_rows,columns=["id","recordDate","temperature"])
weather_pd["recordDate"]=pd.to_datetime(weather_pd["recordDate"])
weather_pd

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
spark=SparkSession.builder.getOrCreate()
weather_spark=spark.createDataFrame(weather_rows,["id","recordDate","temperature"]).withColumn("recordDate",F.to_date("recordDate"))
weather_spark.createOrReplaceTempView("Weather")

# 2. SQL Solution

In [ ]:
sql_result=spark.sql("""
SELECT w1.id
FROM Weather w1
JOIN Weather w2
  ON DATEDIFF(w1.recordDate,w2.recordDate)=1
WHERE w1.temperature>w2.temperature
ORDER BY w1.id
""")
sql_result.show(truncate=False)

# 3. pandas Solution

In [ ]:
ordered_pd=weather_pd.sort_values("recordDate").copy()
ordered_pd["prev_date"]=ordered_pd["recordDate"].shift(1)
ordered_pd["prev_temp"]=ordered_pd["temperature"].shift(1)
result_pd=(ordered_pd.loc[((ordered_pd["recordDate"]-ordered_pd["prev_date"]).dt.days==1)&(ordered_pd["temperature"]>ordered_pd["prev_temp"]),["id"]].reset_index(drop=True))
result_pd

# 4. PySpark Solution

In [ ]:
w=Window.orderBy("recordDate")
result_spark=(weather_spark.withColumn("prev_date",F.lag("recordDate").over(w)).withColumn("prev_temp",F.lag("temperature").over(w)).filter((F.datediff("recordDate","prev_date")==1)&(F.col("temperature")>F.col("prev_temp"))).select("id").orderBy("id"))
result_spark.show(truncate=False)

# 5. Pattern Mapping
| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| previous row | self join by date | `.shift()` | `Window + lag()` |
| one-day gap | `DATEDIFF(...)=1` | datetime subtraction | `F.datediff()` |

# 6. Muscle-Memory Round

พิมพ์ใหม่เองโดยไม่ copy คำตอบด้านบน

In [ ]:
# MUSCLE MEMORY — SQL
# Rebuild using temp view(s): Weather

In [ ]:
# MUSCLE MEMORY — PANDAS
# Rebuild using: weather_pd

In [ ]:
# MUSCLE MEMORY — PYSPARK
# Rebuild using: weather_spark